run this headless

conda activate guitarmidi
screen jupyter nbconvert --to notebook --execute traning.ipynb --output=output_notebook.ipynb --ExecutePreprocessor.timeout=-1

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import os
import glob # To list files
import time
from IPython.display import clear_output

# --- Essential for GPU memory management ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("Mixed precision policy set to 'mixed_float16'.")

        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True) # Corrected
        print("Memory growth enabled for GPUs.")
    except RuntimeError as e:
        print(f"Error configuring GPU: {e}")
# ---------------------------------------------------------------------------------

print(f"TensorFlow version: {tf.__version__}")

# --- Configuration (using values from serialization part) ---
image_height = 256
image_width = 312 # Assuming this is your updated 288+some context/padding, or just 288 filter outputs
num_channels = 1
num_classes = 89 # For MIDI notes
INPUT_SHAPE = (image_height, image_width, num_channels)
OUTPUT_DIM_NOTES = num_classes # For notes output
OUTPUT_DIM_ONSETS = 1 # For onsets output
LEARNING_RATE = 0.001
BATCH_SIZE = 16 # Adjust as needed
EPOCHS = 100

# Directories where slices were saved
input_data_dir = 'data_slices/input'
output_data_dir = 'data_slices/output' # This will be for the 'note' labels
onsets_data_dir = 'data_slices/onsets' # This will be for the 'onsets' labels

# --- Custom Callback for Live Loss Plotting (remains the same, but will show two sets of metrics) ---
class JupyterLivePlottingCallback(Callback):
    def __init__(self, fig_title="Training Metrics"):
        super().__init__()
        self.fig_title = fig_title
        # Updated to store metrics for both outputs
        self.epoch_data = {
            'loss': [], 'note_loss': [], 'onsets_loss': [],
            'note_accuracy': [], 'onsets_accuracy': [],
            'val_loss': [], 'val_note_loss': [], 'val_onsets_loss': [],
            'val_note_accuracy': [], 'val_onsets_accuracy': []
        }
        self.epochs = []

    def on_train_begin(self, logs=None):
        self.epoch_data = {k: [] for k in self.epoch_data.keys()}
        self.epochs = []
        print("Starting Keras model training with live plot. Output will update below...")

    def on_epoch_end(self, epoch, logs=None):
        clear_output(wait=True)

        self.epochs.append(epoch + 1)
        for key in self.epoch_data:
            self.epoch_data[key].append(logs.get(key))

        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(self.fig_title)

        # Plot Note Accuracy
        axes[0, 0].plot(self.epochs, self.epoch_data['note_output_accuracy'], 'b-o', label='Training Note Accuracy')
        axes[0, 0].plot(self.epochs, self.epoch_data['val_note_output_accuracy'], 'r-x', label='Validation Note Accuracy')
        axes[0, 0].set_title('Note Accuracy')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Accuracy')
        axes[0, 0].grid(True)
        axes[0, 0].legend(loc='lower right')
        axes[0, 0].set_xticks(self.epochs if len(self.epochs) < 15 else self.epochs[::2])

        # Plot Onsets Accuracy
        axes[0, 1].plot(self.epochs, self.epoch_data['onsets_output_accuracy'], 'b-o', label='Training Onsets Accuracy')
        axes[0, 1].plot(self.epochs, self.epoch_data['val_onsets_output_accuracy'], 'r-x', label='Validation Onsets Accuracy')
        axes[0, 1].set_title('Onsets Accuracy')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].grid(True)
        axes[0, 1].legend(loc='lower right')
        axes[0, 1].set_xticks(self.epochs if len(self.epochs) < 15 else self.epochs[::2])
        
        # Plot Total Loss
        axes[1, 0].plot(self.epochs, self.epoch_data['loss'], 'b-o', label='Total Training Loss')
        axes[1, 0].plot(self.epochs, self.epoch_data['val_loss'], 'r-x', label='Total Validation Loss')
        axes[1, 0].set_title('Total Loss')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Loss')
        axes[1, 0].grid(True)
        axes[1, 0].legend(loc='upper right')
        axes[1, 0].set_xticks(self.epochs if len(self.epochs) < 15 else self.epochs[::2])

        # Plot Individual Losses (Note vs Onsets)
        axes[1, 1].plot(self.epochs, self.epoch_data['note_output_loss'], 'g-o', label='Training Note Loss')
        axes[1, 1].plot(self.epochs, self.epoch_data['val_note_output_loss'], 'g--x', label='Validation Note Loss')
        axes[1, 1].plot(self.epochs, self.epoch_data['onsets_output_loss'], 'm-o', label='Training Onsets Loss')
        axes[1, 1].plot(self.epochs, self.epoch_data['val_onsets_output_loss'], 'm--x', label='Validation Onsets Loss')
        axes[1, 1].set_title('Individual Task Losses')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Loss')
        axes[1, 1].grid(True)
        axes[1, 1].legend(loc='upper right')
        axes[1, 1].set_xticks(self.epochs if len(self.epochs) < 15 else self.epochs[::2])


        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

    def on_train_end(self, logs=None):
        print("Training finished. Final plot above.")


# --- 1. Define the Pure CNN Model (with two outputs) ---
def build_cnn_model(input_shape, output_dim_notes, output_dim_onsets):

    inputs = layers.Input(shape=input_shape, dtype=tf.float32, name='input_features')

    x = layers.Conv2D(filters=32, kernel_size=(3, 3), padding='same', activation=None)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation='relu')(x)
    
    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2))(x)
    x = layers.SpatialDropout2D(0.2)(x)

    x = layers.Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation='relu')(x)
    
    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2))(x)
    x = layers.SpatialDropout2D(0.25)(x)


    x = layers.Conv2D(filters=128, kernel_size=(3, 3), padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation='relu')(x)
    
    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2))(x)
    x = layers.SpatialDropout2D(0.3)(x)

    # --- Branch for Note prediction ---
    note_output_branch = layers.GlobalAveragePooling2D()(x)
    note_output_branch = layers.Dropout(0.4)(note_output_branch)
    note_output = layers.Dense(output_dim_notes, activation='sigmoid', dtype=tf.float32, name='note_output')(note_output_branch)
    
    # --- Branch for Onsets prediction ---
    onsets_output_branch = layers.GlobalAveragePooling2D()(x) # Can share the same pooled features or have a separate branch
    onsets_output_branch = layers.Dropout(0.4)(onsets_output_branch)
    onsets_output = layers.Dense(output_dim_onsets, activation='sigmoid', dtype=tf.float32, name='onsets_output')(onsets_output_branch)

    return Model(inputs=inputs, outputs=[note_output, onsets_output])

# --- 2. Compile the Model (Updated for multiple outputs) ---
cnn_model = build_cnn_model(INPUT_SHAPE, OUTPUT_DIM_NOTES, OUTPUT_DIM_ONSETS)

# Define loss functions for each output
losses = {
    'note_output': 'binary_crossentropy',
    'onsets_output': 'binary_crossentropy'
}

# Define metrics for each output
metrics = {
    'note_output': 'accuracy', # or tf.keras.metrics.BinaryAccuracy()
    'onsets_output': 'accuracy' # or tf.keras.metrics.BinaryAccuracy()
}

# Optional: Assign loss weights if one task is more important or if classes are highly imbalanced
# For onsets, you might want to give it a higher weight because onset events are typically sparse.
loss_weights = {
    'note_output': 1.0,
    'onsets_output': 5.0 # Give onset prediction more importance, adjust as needed
}

cnn_model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss=losses,
                  loss_weights=loss_weights, # Include loss weights
                  metrics=metrics)

cnn_model.summary()

# --- 3. Data Loading and Preparation (Stream from Disk - .npy files) ---

# Get lists of all input and output file paths
input_filepaths = sorted(glob.glob(os.path.join(input_data_dir, '*.npy')))
output_filepaths = sorted(glob.glob(os.path.join(output_data_dir, '*.npy'))) # For notes
onsets_filepaths = sorted(glob.glob(os.path.join(onsets_data_dir, '*.npy'))) # For onsets

total_samples_on_disk = len(input_filepaths)
if total_samples_on_disk == 0:
    print(f"ERROR: No .npy files found in {input_data_dir}. Please run the serialization script first.")
    exit()
if total_samples_on_disk != len(output_filepaths) or total_samples_on_disk != len(onsets_filepaths):
    print("ERROR: Mismatch in number of input, note output, or onsets output files.")
    exit()

print(f"Found {total_samples_on_disk} files on disk.")
class_weights_onsets = {
    0.0: 0.1,  # Lower weight for Class 0 (e.g., the majority class)
    0.25:1.0,
    0.5:1.0,
    1.0: 2.0   # Higher weight for Class 1 (e.g., the minority class)
}

print("Class Weights for onsets_output:")
print(class_weights_onsets)

# Function to load a single image and its labels from file paths (Updated)
def load_input_from_files(input_path_tensor):
    # Convert TensorFlow string tensors to Python strings
    input_path = input_path_tensor.numpy().decode('utf-8')

    
    # Load the NumPy arrays
    image = np.load(input_path).astype(np.float32).reshape(INPUT_SHAPE)

    image = tf.ensure_shape(image, INPUT_SHAPE)

  
    return image


def load_notes_from_files( note_output_path_tensor):
    # Convert TensorFlow string tensors to Python strings

    note_output_path = note_output_path_tensor.numpy().decode('utf-8')

    
    # Load the NumPy arrays

    note_label = np.load(note_output_path).astype(np.float32).reshape(OUTPUT_DIM_NOTES)

    # Ensure shapes for TensorFlow

    note_label = tf.ensure_shape(note_label, (OUTPUT_DIM_NOTES,))

  
    return  note_label

def load_onsets_from_files(onset_path_tensor):
    # Convert TensorFlow string tensors to Python strings

    onset_path = onset_path_tensor.numpy().decode('utf-8')
    
    # Load the NumPy arrays

    onset_label = np.load(onset_path).astype(np.float32).reshape(OUTPUT_DIM_ONSETS)
    onset_weights=np.vectorize(class_weights_onsets.get)(onset_label)
    # onset_locations=np.where(onset_label==0.25)[0]
    # if(len(onset_locations)>0):
    #     loc=onset_locations[0]
    #     print('onset at index '+str(loc)+' weight: '+str(onset_weights[loc]))
    # Ensure shapes for TensorFlow

    onset_label = tf.ensure_shape(onset_label, (OUTPUT_DIM_ONSETS,))
    onset_weights = tf.convert_to_tensor(onset_weights, dtype=tf.float32)
    onset_weights=tf.ensure_shape(onset_weights, (OUTPUT_DIM_ONSETS,))
  
    return  onset_label,onset_weights



# TensorFlow wrapper function (Updated)
def tf_load_sample_from_files(ipath, nopath, opath):
    image = tf.py_function(
        load_input_from_files, [ipath], [tf.float32]
    )[0]
    note_label = tf.py_function(
        load_notes_from_files, [ nopath], [tf.float32]
    )[0]
    
    onset_label,onset_weights = tf.py_function(
        load_onsets_from_files, [opath], [tf.float32,tf.float32]
    )

    # Explicitly set shapes here!
    image.set_shape(INPUT_SHAPE)
    note_label.set_shape((OUTPUT_DIM_NOTES,))
    onset_label.set_shape((OUTPUT_DIM_ONSETS,))
    onset_weights.set_shape((OUTPUT_DIM_ONSETS,))
    #onsets_sample_weight.set_shape(()) # Scalar weight
    
    # Return as (input_data, (output1_labels, output2_labels))
    return (image, (note_label,onset_label),(None,onset_weights)
            )

# Create a dataset from the lists of file paths
dataset = tf.data.Dataset.from_tensor_slices((input_filepaths, output_filepaths, onsets_filepaths))

# Shuffle the list of paths first
dataset = dataset.shuffle(buffer_size=total_samples_on_disk) # Buffer size for shuffling paths

# Split the dataset into training and validation subsets based on indices
split_ratio = 0.7
num_train = int(total_samples_on_disk * split_ratio)

train_dataset = dataset.take(num_train)



val_dataset = dataset.skip(num_train)

# Map the loading function to the datasets using tf.py_function
train_dataset = train_dataset.map(
    tf_load_sample_from_files,
    num_parallel_calls=tf.data.AUTOTUNE # Load in parallel threads/processes
)
val_dataset = val_dataset.map(
    tf_load_sample_from_files,
    num_parallel_calls=tf.data.AUTOTUNE
)

# Apply batching and prefetching
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --- 3. Configure Callbacks (Updated monitor names) ---
early_stopping_acc = EarlyStopping(
    monitor='val_note_output_accuracy', # Monitor validation accuracy for the 'note_output'
    patience=10,            # Number of epochs with no improvement
    mode='max',             # 'max' because we want to maximize accuracy
    verbose=1,              # Log when training stops
    restore_best_weights=True # Restore weights from the epoch with the best monitored value.
)

model_checkpoint_acc = ModelCheckpoint(
    'best_model_by_note_acc.keras', # A different filename
    monitor='val_note_output_accuracy',
    mode='max',
    save_best_only=True, # Only save when validation accuracy improves
    verbose=1
)

jupyter_live_plot = JupyterLivePlottingCallback(fig_title="Keras Training Progress (Multi-Output)")

# --- 4. Training the Model ---
print("\n--- Training Pure CNN Model (Multi-Output) ---")
try:
    history_cnn = cnn_model.fit(train_dataset,
                                epochs=EPOCHS,
                                validation_data=val_dataset,
                                callbacks=[model_checkpoint_acc, early_stopping_acc, jupyter_live_plot])
    # You might want to save the final weights too, or rely on ModelCheckpoint
    cnn_model.save_weights('guitarmidi-multi-output-final.weights.h5')
    print("Final model weights saved successfully!")
except Exception as e:
    print(f"An error occurred during training: {e}")

Mixed precision policy set to 'mixed_float16'.
Memory growth enabled for GPUs.
TensorFlow version: 2.19.0


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_features      │ (None, 256, 312,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 256, 312,  │        320 │ input_features[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 312,  │        128 │ conv2d_9[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_9        │ (None, 256, 312,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, 128, 156,  │          0 │ activation_9[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d_9 │ (None, 128, 156,  │          0 │ max_pooling2d_9[… │
│ (SpatialDropout2D)  │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 128, 156,  │     18,496 │ spatial_dropout2… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 156,  │        256 │ conv2d_10[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_10       │ (None, 128, 156,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_10    │ (None, 64, 78,    │          0 │ activation_10[0]… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d_… │ (None, 64, 78,    │          0 │ max_pooling2d_10… │
│ (SpatialDropout2D)  │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 64, 78,    │     73,856 │ spatial_dropout2… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 78,    │        512 │ conv2d_11[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_11       │ (None, 64, 78,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_11    │ (None, 32, 39,    │          0 │ activation_11[0]… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d_… │ (None, 32, 39,    │          0 │ max_pooling2d_11… │
│ (SpatialDropout2D)  │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ spatial_dropout2

 Total params: 105,178 (410.85 KB)

 Trainable params: 104,730 (409.10 KB)

 Non-trainable params: 448 (1.75 KB)

Found 196500 files on disk.
Class Weights for onsets_output:
{0.0: 0.1, 0.25: 1.0, 0.5: 1.0, 1.0: 2.0}

--- Training Pure CNN Model (Multi-Output) ---
Starting Keras model training with live plot. Output will update below...
Epoch 1/100
onset at index 0 weight: 1.0
   3/8597 ━━━━━━━━━━━━━━━━━━━━ 5:31 39ms/step - loss: 1.0930 - note_output_accuracy: 0.0556 - note_output_loss: 0.7968 - onsets_output_accuracy: 0.7431 - onsets_output_loss: 0.0592  

2025-07-03 16:13:33.949780: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_fusion_8', 8 bytes spill stores, 8 bytes spill loads



   6/8597 ━━━━━━━━━━━━━━━━━━━━ 7:49 55ms/step - loss: 1.0916 - note_output_accuracy: 0.0406 - note_output_loss: 0.7763 - onsets_output_accuracy: 0.7892 - onsets_output_loss: 0.0631onset at index 0 weight: 1.0
  15/8597 ━━━━━━━━━━━━━━━━━━━━ 8:00 56ms/step - loss: 1.2063 - note_output_accuracy: 0.0345 - note_output_loss: 0.7187 - onsets_output_accuracy: 0.8304 - onsets_output_loss: 0.0975onset at index 0 weight: 1.0
  17/8597 ━━━━━━━━━━━━━━━━━━━━ 8:02 56ms/step - loss: 1.2031 - note_output_accuracy: 0.0340 - note_output_loss: 0.7072 - onsets_output_accuracy: 0.8341 - onsets_output_loss: 0.0992onset at index 0 weight: 1.0
onset at index 0 weight: 1.0
  19/8597 ━━━━━━━━━━━━━━━━━━━━ 8:01 56ms/step - loss: 1.1954 - note_output_accuracy: 0.0333 - note_output_loss: 0.6961 - onsets_output_accuracy: 0.8377 - onsets_output_loss: 0.0999onset at index 0 weight: 1.0
  24/8597 ━━━━━━━━━━━━━━━━━━━━ 7:57 56ms/step - loss: 1.1696 - note_output_accuracy: 0.0315 - note_output_loss: 0.6701 - onsets_output_

KeyboardInterrupt: 

In [ ]:
cnn_model.save_weights('guitarmidi-40-dropout.weights.h5')